# MASI Kaggle Medium Run From GitHub

This notebook is the default Kaggle entrypoint for the prepared subset workflow.

It does all of the following:
- validates the required prepared subset dataset before doing expensive work,
- clones the MASI repository into `/kaggle/working/MASI`,
- installs the repository with `pip install -e .[recommender]`,
- optionally restores a prior resume bundle into `/kaggle/working/masi_artifacts`,
- validates attached subset images and downloads only missing ones into the writable cache,
- runs the bounded Phase 1 through Phase 3 pipeline via `scripts/train_masi.py`,
- verifies the expected manifests and checkpoint directories,
- exports a dataset-ready bundle for a later Kaggle session.

Required Kaggle inputs:
- a prepared dataset mounted under a slug like `masi-amazon-csj-subset-medium`
- files `Clothing_Shoes_and_Jewelry.jsonl`, `meta_Clothing_Shoes_and_Jewelry.jsonl`, `subset_manifest.json`, and `images/`
- optionally, a second resume dataset mounted at `/kaggle/input/<resume-slug>`


In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
DATASET_SLUG = "masi-amazon-csj-subset-medium"
RESUME_DATASET_SLUG = None

REPO_URL = "https://github.com/pradyunuydarp/MASI.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/MASI")

STORAGE_ROOT = Path("/kaggle/working/masi_artifacts")
RUN_NAME = "amazon_csj_subset_medium_train"
RUN_ROOT = STORAGE_ROOT / "outputs" / RUN_NAME
CACHE_ROOT = STORAGE_ROOT / "data" / "processed" / "amazon_csj_subset_medium"
CONFIG_PATH = Path("configs/masi_train_csj_subset_medium.json")

def find_dataset_dir(slug: str | None) -> Path | None:
    if not slug:
        return None
    candidates = []
    direct_candidate = INPUT_ROOT / slug
    if direct_candidate.is_dir():
        candidates.append(direct_candidate)
    datasets_root = INPUT_ROOT / "datasets"
    if datasets_root.is_dir():
        candidates.extend(path for path in sorted(datasets_root.glob(f"*/{slug}")) if path.is_dir())
    candidates.extend(path for path in sorted(INPUT_ROOT.rglob(slug)) if path.is_dir())
    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        return resolved
    return None

DATASET_DIR = find_dataset_dir(DATASET_SLUG)
RESUME_DATASET_DIR = find_dataset_dir(RESUME_DATASET_SLUG)

print(f"Input root: {INPUT_ROOT}")
print(f"Prepared dataset slug: {DATASET_SLUG}")
print(f"Prepared dataset dir: {DATASET_DIR}")
print(f"Resume dataset dir: {RESUME_DATASET_DIR}")
print(f"Repo URL: {REPO_URL}")
print(f"Repo branch: {REPO_BRANCH}")
print(f"Repo dir: {REPO_DIR}")
print(f"Storage root: {STORAGE_ROOT}")
print(f"Run root: {RUN_ROOT}")
print(f"Cache root: {CACHE_ROOT}")
print(f"Config path inside repo: {CONFIG_PATH}")


In [ ]:
from pathlib import Path

if not INPUT_ROOT.exists():
    raise FileNotFoundError(f"Kaggle input root does not exist: {INPUT_ROOT}")

mounted_inputs = sorted(path.name for path in INPUT_ROOT.iterdir())
print("Mounted Kaggle inputs:")
for name in mounted_inputs:
    print(f"- {name}")

if DATASET_DIR is None or not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Attach the prepared subset dataset '{DATASET_SLUG}' so the notebook can read bounded CSJ inputs."
    )

required_paths = [
    DATASET_DIR / "Clothing_Shoes_and_Jewelry.jsonl",
    DATASET_DIR / "meta_Clothing_Shoes_and_Jewelry.jsonl",
    DATASET_DIR / "subset_manifest.json",
    DATASET_DIR / "images",
]
for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required prepared subset artifact: {required_path}")

if RESUME_DATASET_DIR is not None and not RESUME_DATASET_DIR.exists():
    raise FileNotFoundError(f"Resume dataset path does not exist: {RESUME_DATASET_DIR}")

print(f"Validated prepared dataset: {DATASET_DIR}")
if RESUME_DATASET_DIR is None:
    print("No resume dataset configured.")
else:
    print(f"Validated resume dataset: {RESUME_DATASET_DIR}")


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ],
    check=True,
)

resolved_config_path = REPO_DIR / CONFIG_PATH
if not resolved_config_path.exists():
    raise FileNotFoundError(f"Missing expected config after clone: {resolved_config_path}")

STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

print(f"Cloned repo into {REPO_DIR}")
print(f"Resolved config path: {resolved_config_path}")
print(f"Storage root: {STORAGE_ROOT}")

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[recommender]"], check=True)


In [ ]:
from pathlib import Path
import shutil

if RESUME_DATASET_DIR is None:
    print("No resume dataset attached; starting from a fresh working directory.")
else:
    for source_path in sorted(RESUME_DATASET_DIR.iterdir()):
        destination_path = STORAGE_ROOT / source_path.name
        if destination_path.exists():
            if destination_path.is_dir():
                shutil.rmtree(destination_path)
            else:
                destination_path.unlink()
        if source_path.is_dir():
            shutil.copytree(source_path, destination_path)
        else:
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_path)
    print(f"Restored prior artifacts from {RESUME_DATASET_DIR} into {STORAGE_ROOT}")


## Optional Image Prefetch

This step validates images already attached in the prepared dataset and downloads only missing ones into the writable cache.


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run(
    [
        sys.executable,
        "scripts/download_amazon_csj_images.py",
        "--config",
        str(CONFIG_PATH),
        "--storage-root",
        str(STORAGE_ROOT),
        "--workers",
        "8",
        "--retries",
        "2",
        "--resume",
    ],
    check=True,
    env={**os.environ, "PYTHONPATH": "src"},
)


## Run The Bounded MASI Pipeline

This executes the bounded end-to-end path through Phase 1 alignment, Phase 2 dual codebooks, and Phase 3 MLM plus sequential fine-tuning.


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run(
    [
        sys.executable,
        "scripts/train_masi.py",
        "--config",
        str(CONFIG_PATH),
        "--storage-root",
        str(STORAGE_ROOT),
    ],
    check=True,
    env={**os.environ, "PYTHONPATH": "src"},
)


In [ ]:
from pathlib import Path
import json

summary_paths = {
    "image_download_manifest": RUN_ROOT / "image_download_manifest.json",
    "run_manifest": RUN_ROOT / "run_manifest.json",
    "token_summary": RUN_ROOT / "phase12_tokens" / "masi_token_summary.json",
    "experiment_summary": RUN_ROOT / "phase3_experiment" / "experiment_summary.json",
}

required_labels = ["run_manifest", "token_summary", "experiment_summary"]
for label in required_labels:
    path = summary_paths[label]
    if not path.exists():
        raise FileNotFoundError(f"Expected required artifact was not created: {path}")

for label, path in summary_paths.items():
    print(f"\n=== {label}: {path} ===")
    if not path.exists():
        print("missing")
        continue
    payload = json.loads(path.read_text())
    print(json.dumps(payload, indent=2)[:4000])


In [ ]:
phase12_checkpoint_root = RUN_ROOT / "checkpoints" / "phase12_tokens"
phase3_checkpoint_root = RUN_ROOT / "checkpoints" / "phase3_experiment"

print("Phase 1/2 checkpoints:")
if phase12_checkpoint_root.exists():
    for path in sorted(phase12_checkpoint_root.rglob("*")):
        print(path)
else:
    print(f"missing: {phase12_checkpoint_root}")

print("\nPhase 3 checkpoints:")
if phase3_checkpoint_root.exists():
    for path in sorted(phase3_checkpoint_root.rglob("*")):
        print(path)
else:
    print(f"missing: {phase3_checkpoint_root}")


In [ ]:
from pathlib import Path
import shutil
import subprocess

EXPORT_ROOT = Path("/kaggle/working/MASI_subset_medium_export")
ZIP_PATH = Path("/kaggle/working/MASI_subset_medium_export.zip")

if not RUN_ROOT.exists():
    raise FileNotFoundError(f"Run output directory does not exist yet: {RUN_ROOT}")

if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

export_run_root = EXPORT_ROOT / "outputs" / RUN_ROOT.name
export_run_root.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_ROOT, export_run_root)

if CACHE_ROOT.exists():
    export_cache_root = EXPORT_ROOT / "data" / "processed" / CACHE_ROOT.name
    export_cache_root.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(CACHE_ROOT, export_cache_root)
else:
    print(f"Skipping cache export because {CACHE_ROOT} does not exist")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
subprocess.run(["zip", "-r", str(ZIP_PATH), str(EXPORT_ROOT)], check=True)
print(f"Wrote {ZIP_PATH}")
print("Upload the export directory or zip as a private Kaggle Dataset to resume a later session.")
